# SageMaker Model Registry with Boto3
## A cell-by-cell companion to Class 15 · US West (Oregon)

Follow the same hotel-cancellation example as the manual notebook: **register v1, approve it,
deploy it, release v2, check its predictions, and roll back**.

Each cell shows the AWS call for the step immediately above it. Run from top to bottom.
The two trained models and the 1,000 labelled bookings are the same as in the main notebook.

| In the manual notebook | In this notebook |
|---|---|
| Upload the model files in S3 | `s3.upload_file(...)` |
| Register a group and model versions | `create_model_package_group(...)`, `create_model_package(...)` |
| Change approval and lifecycle | `update_model_package(...)` |
| Deploy a registered version | `create_model(...)`, `create_endpoint_config(...)`, `create_endpoint(...)` |
| Send bookings to the model | `invoke_endpoint(...)` |
| Change the serving model or roll back | `update_endpoint(...)` |
| Tear down | The corresponding `delete_...` calls |

## 1. Setup

Use an authenticated Python notebook: SageMaker Studio or local Jupyter with AWS credentials
already configured. You need an existing **SageMaker execution role**, an **S3 bucket in
`us-west-2`**, permission to manage the resources below and pass that role, and sufficient
`ml.m5.large` endpoint quota. The execution role must be able to read the model files.

Set `ROLE_ARN` in the configuration cell. Set `BUCKET` if your bucket has a different name
from `sagemaker-us-west-2-<account-id>`. Keep credentials outside the notebook.

This demo uses a **single-model endpoint**. The main notebook's Studio Deploy button created
an inference component. Here, `update_endpoint` switches configurations, so calls do not need
an inference-component name. AWS starts the replacement fleet before retiring the old one.

**Run once, top to bottom.** Rerunning creation cells can create another version or encounter an
existing resource. Clean up before starting a new run. Endpoint instances incur charges while
provisioned; an update can temporarily use an additional instance. Run section 9 before leaving.
The S3 cleanup assumes an unversioned demo bucket; a versioned bucket retains older object versions.

In [ ]:
%pip install -q "boto3==1.44.3" "numpy>=1.26,<3" "pandas>=2.2,<3"

Import the libraries and choose the region and release policy. The 15% false-positive limit
and 50% minimum recall are **classroom assumptions**, not measured hotel requirements.
They make the release decision explicit. A booking is flagged when its probability is **greater
than 0.5**, matching the manual notebook.

In [ ]:
import time
import uuid
from pathlib import Path
from urllib.request import urlretrieve

import boto3
import numpy as np
import pandas as pd
from IPython.display import display

REGION = "us-west-2"  # US West (Oregon)
ROLE_ARN = "PASTE_YOUR_SAGEMAKER_EXECUTION_ROLE_ARN_HERE"
BUCKET = ""  # Leave empty to use sagemaker-us-west-2-<account-id>.
INSTANCE_TYPE = "ml.m5.large"
IMAGE_URI = "683313688378.dkr.ecr.us-west-2.amazonaws.com/sagemaker-xgboost:1.7-1"

threshold = 0.5
max_false_positive_rate = 0.15
min_recall = 0.50

Create the clients. `sagemaker` manages registry and hosting resources;
`sagemaker-runtime` sends prediction requests to the endpoint.

In [ ]:
session = boto3.Session(region_name=REGION)
sm = session.client("sagemaker")
runtime = session.client("sagemaker-runtime")
s3 = session.client("s3")

account_id = session.client("sts").get_caller_identity()["Account"]
bucket = BUCKET or f"sagemaker-{REGION}-{account_id}"
print("Region:", REGION)
print("Bucket:", bucket)

Use separate names for this demo so it can run alongside the manual notebook. Keep the printed
names if you restart the kernel; the AWS resources continue to exist after a restart.

In [ ]:
run_id = uuid.uuid4().hex[:8]
group_name = f"hotel-code-{run_id}"
endpoint_name = f"hotel-code-{run_id}-endpoint"
v1_model_name = f"{group_name}-v1"
v2_model_name = f"{group_name}-v2"
v1_config_name = f"{group_name}-v1-config"
v2_config_name = f"{group_name}-v2-config"
s3_prefix = f"hotel-registry-demo/{run_id}"

print("Model group:", group_name)
print("Endpoint:", endpoint_name)
print("S3 prefix:", s3_prefix)

## 2. Download and upload the model files

Download the two models and the replay data. The URL uses a fixed source commit so the lesson
continues to use the same files. The CSV is already encoded in the column order the models expect.

In [ ]:
source_commit = "bee339bc1be89bc76d8b54764f8d4f6eb7d5654e"
repo = f"https://raw.githubusercontent.com/pulkit-scaler/mlops-model-registry/{source_commit}"
local_dir = Path("hotel_registry_demo")
files = [
    "models/v1/model.tar.gz", "models/v2/model.tar.gz",
    "data/live.csv", "data/live_labels.csv",
]

for file in files:
    destination = local_dir / file
    destination.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(f"{repo}/{file}", destination)
    print("Downloaded:", file)

Upload the models to S3, just as you uploaded the two `model.tar.gz` files through the console.
The evaluation CSV files stay local.

In [ ]:
v1_s3_key = f"{s3_prefix}/v1/model.tar.gz"
v2_s3_key = f"{s3_prefix}/v2/model.tar.gz"

s3.upload_file(str(local_dir / "models/v1/model.tar.gz"), bucket, v1_s3_key)
s3.upload_file(str(local_dir / "models/v2/model.tar.gz"), bucket, v2_s3_key)

v1_s3_uri = f"s3://{bucket}/{v1_s3_key}"
v2_s3_uri = f"s3://{bucket}/{v2_s3_key}"
print(v1_s3_uri)
print(v2_s3_uri)

## 3. Register version 1

First create the group. A group collects versions of the same model/problem.

In [ ]:
sm.create_model_package_group(
    ModelPackageGroupName=group_name,
    ModelPackageGroupDescription="Hotel cancellation models — code demo",
)

Now register the v1 artifact. The package holds its S3 address, serving image and supplied
validation AUC. It starts as `PendingManualApproval`. Save the returned ARN; that identifies
the exact registered version in later calls.

In [ ]:
response = sm.create_model_package(
    ModelPackageGroupName=group_name,
    ModelPackageDescription="v1: 100 trees, depth 4",
    InferenceSpecification={
        "Containers": [{"Image": IMAGE_URI, "ModelDataUrl": v1_s3_uri}],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
    },
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={"validation_auc": "0.8817"},
)
v1_package_arn = response["ModelPackageArn"]
print(v1_package_arn)

Wait for registration to complete. This small loop is visible here because there is no Boto3
model-package waiter equivalent to the endpoint waiter used later.

In [ ]:
v1_package = sm.describe_model_package(ModelPackageName=v1_package_arn)
while v1_package["ModelPackageStatus"] in ("Pending", "InProgress"):
    time.sleep(3)
    v1_package = sm.describe_model_package(ModelPackageName=v1_package_arn)

assert v1_package["ModelPackageStatus"] == "Completed"
print("Registration:", v1_package["ModelPackageStatus"])
print("Approval:", v1_package["ModelApprovalStatus"])

## 4. Approve version 1

Move v1 into `PreProduction` and approve it for deployment. These are separate fields, even
though one API call can update both. `Approved` permits deployment; `PreProduction` records
where the version is in the release process.

In [ ]:
sm.update_model_package(
    ModelPackageArn=v1_package_arn,
    ModelApprovalStatus="Approved",
    ApprovalDescription="First version; validation AUC 0.8817",
    ModelLifeCycle={
        "Stage": "PreProduction",
        "StageStatus": "InProgress",
        "StageDescription": "Preparing the first release",
    },
)

## 5. Deploy version 1

### 5.1 Create a SageMaker model

The deployable model refers to the **package ARN**, so SageMaker obtains the artifact and
container from that registered version.

In [ ]:
sm.create_model(
    ModelName=v1_model_name,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={"ModelPackageName": v1_package_arn},
)

### 5.2 Create an endpoint configuration

The configuration selects the model and the machine serving it. `AllTraffic` is the name of
the single production variant; all requests go to it.

In [ ]:
sm.create_endpoint_config(
    EndpointConfigName=v1_config_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": v1_model_name,
        "InstanceType": INSTANCE_TYPE,
        "InitialInstanceCount": 1,
        "InitialVariantWeight": 1.0,
    }],
)

### 5.3 Create the endpoint and wait

This starts the hosted compute. The standard Boto3 waiter polls until the endpoint is
`InService`; this usually takes several minutes. If AWS reports that the new configuration
is not yet visible, wait a few seconds and rerun this cell.

In [ ]:
sm.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=v1_config_name,
)

sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint_name)
print("Endpoint is InService:", endpoint_name)

### 5.4 Check which registered version is serving

Follow the same trail that the manual notebook followed in the console:
**endpoint → endpoint configuration → model → package ARN**.

In [ ]:
endpoint = sm.describe_endpoint(EndpointName=endpoint_name)
config = sm.describe_endpoint_config(EndpointConfigName=endpoint["EndpointConfigName"])
model = sm.describe_model(ModelName=config["ProductionVariants"][0]["ModelName"])

print("Serving package:", model["PrimaryContainer"]["ModelPackageName"])
print("Expected package:", v1_package_arn)

### 5.5 Send the 1,000 bookings

These are historical bookings whose outcomes are already known. This is a replay evaluation;
it does not simulate waiting for new guests' eventual cancellation outcomes.

In [ ]:
replay_csv = (local_dir / "data/live.csv").read_bytes()
labels = pd.read_csv(local_dir / "data/live_labels.csv", header=None)[0].to_numpy()

print("Bookings:", len(labels))
print("Actually cancelled:", int((labels == 1).sum()))

In [ ]:
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Accept="text/csv",
    Body=replay_csv,
)
predictions_v1 = np.fromstring(response["Body"].read().decode(), sep=" ")
assert len(predictions_v1) == len(labels)
print("First three predictions:", predictions_v1[:3])

Count the flagged bookings, true positives and false positives. False-positive rate divides
by guests who **actually came**. Recall divides by bookings that **actually cancelled**.

In [ ]:
flagged_v1 = predictions_v1 > threshold
false_positives_v1 = int((flagged_v1 & (labels == 0)).sum())
true_positives_v1 = int((flagged_v1 & (labels == 1)).sum())
fpr_v1 = false_positives_v1 / (labels == 0).sum()
recall_v1 = true_positives_v1 / (labels == 1).sum()

print("Flagged:", int(flagged_v1.sum()))
print("False positives:", false_positives_v1)
print(f"False-positive rate: {fpr_v1:.2%}")
print(f"Recall: {recall_v1:.2%}")

The reference result is **239 flagged, 78 false positives, 11.11% false-positive rate and
54.03% recall**. Check that v1 meets the classroom policy, then mark it Production.
Unlike the manual sequence, the serving version is checked before recording that stage.

In [ ]:
assert fpr_v1 <= max_false_positive_rate and recall_v1 >= min_recall

sm.update_model_package(
    ModelPackageArn=v1_package_arn,
    ModelLifeCycle={
        "Stage": "Production",
        "StageStatus": "Approved",
        "StageDescription": "First release: endpoint and booking check passed",
    },
)

## 6. Release version 2

### 6.1 Register v2 in the same group

This repeats the registration call with v2's artifact and AUC. There is no retraining.

In [ ]:
response = sm.create_model_package(
    ModelPackageGroupName=group_name,
    ModelPackageDescription="v2: 300 trees, depth 6, cancellations weighted 3x",
    InferenceSpecification={
        "Containers": [{"Image": IMAGE_URI, "ModelDataUrl": v2_s3_uri}],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
    },
    ModelApprovalStatus="PendingManualApproval",
    CustomerMetadataProperties={"validation_auc": "0.9028"},
)
v2_package_arn = response["ModelPackageArn"]
print(v2_package_arn)

In [ ]:
v2_package = sm.describe_model_package(ModelPackageName=v2_package_arn)
while v2_package["ModelPackageStatus"] in ("Pending", "InProgress"):
    time.sleep(3)
    v2_package = sm.describe_model_package(ModelPackageName=v2_package_arn)

assert v2_package["ModelPackageStatus"] == "Completed"
print("Registration:", v2_package["ModelPackageStatus"])

### 6.2 Compare validation AUC and approve v2

Reproduce the manual notebook's initial decision: admit v2 because its reported validation
AUC is higher. The later booking check will show why this admission rule is insufficient.

In [ ]:
v1_package = sm.describe_model_package(ModelPackageName=v1_package_arn)
v2_package = sm.describe_model_package(ModelPackageName=v2_package_arn)
auc_v1 = float(v1_package["CustomerMetadataProperties"]["validation_auc"])
auc_v2 = float(v2_package["CustomerMetadataProperties"]["validation_auc"])

print("v1 AUC:", auc_v1)
print("v2 AUC:", auc_v2)
assert auc_v2 > auc_v1

In [ ]:
sm.update_model_package(
    ModelPackageArn=v2_package_arn,
    ModelApprovalStatus="Approved",
    ApprovalDescription="Validation AUC improved from 0.8817 to 0.9028",
    ModelLifeCycle={
        "Stage": "PreProduction",
        "StageStatus": "InProgress",
        "StageDescription": "Approved for this classroom release; booking check still pending",
    },
)

### 6.3 Create v2's model and endpoint configuration

The same two calls now use v2's package ARN and names. Keep v1's model and configuration:
they are the rollback target.

In [ ]:
sm.create_model(
    ModelName=v2_model_name,
    ExecutionRoleArn=ROLE_ARN,
    PrimaryContainer={"ModelPackageName": v2_package_arn},
)

In [ ]:
sm.create_endpoint_config(
    EndpointConfigName=v2_config_name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": v2_model_name,
        "InstanceType": INSTANCE_TYPE,
        "InitialInstanceCount": 1,
        "InitialVariantWeight": 1.0,
    }],
)

### 6.4 Remember the previous configuration, then update the endpoint

The endpoint name stays the same. The configuration changes from v1 to v2. Saving the previous
configuration means rollback can restore exactly what was serving before this update.

In [ ]:
previous_config_name = sm.describe_endpoint(
    EndpointName=endpoint_name
)["EndpointConfigName"]
print("Rollback target:", previous_config_name)

In [ ]:
sm.update_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=v2_config_name,
)
sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint_name)

active_config = sm.describe_endpoint(EndpointName=endpoint_name)["EndpointConfigName"]
assert active_config == v2_config_name
print("Now serving v2's configuration:", active_config)

## 7. Evaluate v2 and roll back

### 7.1 Repeat the same booking check

The inference call is written out again. It now reaches v2 through the same endpoint name.

In [ ]:
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Accept="text/csv",
    Body=replay_csv,
)
predictions_v2 = np.fromstring(response["Body"].read().decode(), sep=" ")
assert len(predictions_v2) == len(labels)
print("First three predictions:", predictions_v2[:3])

In [ ]:
flagged_v2 = predictions_v2 > threshold
false_positives_v2 = int((flagged_v2 & (labels == 0)).sum())
true_positives_v2 = int((flagged_v2 & (labels == 1)).sum())
fpr_v2 = false_positives_v2 / (labels == 0).sum()
recall_v2 = true_positives_v2 / (labels == 1).sum()

print("Flagged:", int(flagged_v2.sum()))
print("False positives:", false_positives_v2)
print(f"False-positive rate: {fpr_v2:.2%}")
print(f"Recall: {recall_v2:.2%}")

Compare the measured results. V2 catches more actual cancellations, but it also flags many
more guests who arrive. Whether that tradeoff is acceptable depends on the release policy.

In [ ]:
comparison = pd.DataFrame({
    "Model": ["v1", "v2"],
    "Validation AUC": [auc_v1, auc_v2],
    "Flagged": [int(flagged_v1.sum()), int(flagged_v2.sum())],
    "False positives": [false_positives_v1, false_positives_v2],
    "True positives": [true_positives_v1, true_positives_v2],
    "False-positive rate": [fpr_v1, fpr_v2],
    "Recall": [recall_v1, recall_v2],
})
display(comparison.round(4))

### 7.2 Turn the result into a decision

The release passes only if **both** policy limits are met. For the supplied v2, recall passes
but the false-positive rate is about 34.33%, above the 15% limit.

In [ ]:
candidate_passes = (fpr_v2 <= max_false_positive_rate) and (recall_v2 >= min_recall)
print("False-positive rate passes:", fpr_v2 <= max_false_positive_rate)
print("Recall passes:", recall_v2 >= min_recall)
print("Decision:", "KEEP V2" if candidate_passes else "ROLL BACK TO V1")

### 7.3 Restore the endpoint if the check fails

This `if` statement automates the decision previously made by a person. Changing a registry
label alone would not change the serving model; `update_endpoint` performs the rollback.

In [ ]:
if not candidate_passes:
    sm.update_endpoint(
        EndpointName=endpoint_name,
        EndpointConfigName=previous_config_name,
    )
    sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint_name)
    print("Restored v1's endpoint configuration.")
else:
    print("V2 passed; keep its endpoint configuration.")

### 7.4 Update the registry to record the outcome

After a failed check, mark v2 Rejected and Archived. If it passes, mark it Approved and Production.
The next cell keeps v1 approved so that its registered package remains available for rollback.

In [ ]:
sm.update_model_package(
    ModelPackageArn=v2_package_arn,
    ModelApprovalStatus="Approved" if candidate_passes else "Rejected",
    ApprovalDescription="Booking policy passed" if candidate_passes else "Booking policy failed",
    ModelLifeCycle={
        "Stage": "Production" if candidate_passes else "Archived",
        "StageStatus": "Approved" if candidate_passes else "Rejected",
        "StageDescription": "Released" if candidate_passes else "Rolled back to v1",
    },
)

In [ ]:
sm.update_model_package(
    ModelPackageArn=v1_package_arn,
    ModelApprovalStatus="Approved",
    ModelLifeCycle={
        "Stage": "Archived" if candidate_passes else "Production",
        "StageStatus": "Approved",
        "StageDescription": "Replaced by v2" if candidate_passes else "Restored after v2 failed the booking check",
    },
)

## 8. Verify what is serving

Follow the endpoint-to-package trail again. The registry and the actual serving package should
now agree. In this demo, the result should point back to v1.

In [ ]:
endpoint = sm.describe_endpoint(EndpointName=endpoint_name)
config = sm.describe_endpoint_config(EndpointConfigName=endpoint["EndpointConfigName"])
model = sm.describe_model(ModelName=config["ProductionVariants"][0]["ModelName"])
serving_package = model["PrimaryContainer"]["ModelPackageName"]
expected_package = v2_package_arn if candidate_passes else v1_package_arn

print("Serving package:", serving_package)
assert serving_package == expected_package

Send the bookings once more. After rollback, the predictions should match the earlier v1
predictions. This checks the endpoint's answers as well as its configuration.

In [ ]:
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Accept="text/csv",
    Body=replay_csv,
)
final_predictions = np.fromstring(response["Body"].read().decode(), sep=" ")
expected_predictions = predictions_v2 if candidate_passes else predictions_v1

print("Predictions match the expected model:", np.allclose(final_predictions, expected_predictions))
print("False positives:", int(((final_predictions > threshold) & (labels == 0)).sum()))

Read the final approval and lifecycle fields directly from the registry.

In [ ]:
for label, package_arn in [("v1", v1_package_arn), ("v2", v2_package_arn)]:
    package = sm.describe_model_package(ModelPackageName=package_arn)
    print(label,
          "| Approval:", package["ModelApprovalStatus"],
          "| Stage:", package["ModelLifeCycle"]["Stage"])

## 9. Tear down

After completing the demo, delete the endpoint first and wait until it is gone. Then remove
the configurations, models, registry versions, group and S3 artifacts. The existing bucket,
execution role and Studio domain remain available for another class.

These cells assume the demo above completed. If you stopped partway, remove only the resources
you created, using the printed names if necessary. Stop your Studio notebook app separately
when finished.

In [ ]:
sm.delete_endpoint(EndpointName=endpoint_name)
sm.get_waiter("endpoint_deleted").wait(EndpointName=endpoint_name)
print("Endpoint deleted.")

Delete both endpoint configurations and both deployable models.

In [ ]:
for config_name in [v1_config_name, v2_config_name]:
    sm.delete_endpoint_config(EndpointConfigName=config_name)

for model_name in [v1_model_name, v2_model_name]:
    sm.delete_model(ModelName=model_name)

Delete the registered versions. Wait until the group is empty, then delete the group.

In [ ]:
sm.delete_model_package(ModelPackageName=v1_package_arn)
sm.delete_model_package(ModelPackageName=v2_package_arn)

while sm.list_model_packages(ModelPackageGroupName=group_name)["ModelPackageSummaryList"]:
    time.sleep(3)

sm.delete_model_package_group(ModelPackageGroupName=group_name)

Remove only the two model files uploaded by this run.

In [ ]:
for key in [v1_s3_key, v2_s3_key]:
    s3.delete_object(Bucket=bucket, Key=key)

print("Demo cleanup complete.")

## What this demonstrated

- The registry stores model versions and approval/lifecycle records.
- Deployment uses a specific registered package; registry labels do not redirect endpoint traffic.
- Boto3 creates the serving resources, updates the endpoint and sends prediction requests.
- Ordinary Python comparisons and an `if` statement can make the release/rollback decision.
- Rollback reuses the previous endpoint configuration; it does not require retraining or registration.

The next improvement would be to run the booking check on a staging endpoint **before** changing
production. Here it deliberately runs after release to reproduce the main notebook's rollback
story. The same replay is useful for this lesson, but it is not a new independent test set.

## References

- [The fixed class source and model files](https://github.com/pulkit-scaler/mlops-model-registry/tree/bee339bc1be89bc76d8b54764f8d4f6eb7d5654e).
- [AWS: deploying a registered model with Boto3](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry-deploy.html).
- [AWS: updating model approval and lifecycle](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_UpdateModelPackage.html).
- [AWS: updating an endpoint](https://docs.aws.amazon.com/sagemaker/latest/APIReference/API_UpdateEndpoint.html).

**Validation note:** code syntax, notebook structure and the demo's API requests and control flow
were checked offline. Source-model predictions reproduce the original lesson's counts.
This notebook has not been deployed to AWS during authoring.